# MESD (Mexican Emotional Speech Database) -> RGB spectrograms + metadata CSV

Downloads MESD via kagglehub, converts each wav into 3s segments as a mel-spectrogram
RGB image, and appends its metadata to the same `data/labels.csv` and
`data/spectrograms/labels.csv` as the other datasets.

Dataset limitation: the filename only gives the voice TYPE (F/M/C), not an
individual identifier (3 women, 2 men, 6 children exist, but indistinguishable from
each other). `speaker` therefore groups by voice type, not by actual person (keep this in mind
for a speaker-based split). `age` stays `None` (only per-category averages are
known, not individual age).

In [ ]:
%pip install -q librosa matplotlib numpy pandas soundfile pillow nbstripout kagglehub

import sys
!{sys.executable} -m nbstripout --install --attributes .gitattributes

In [ ]:
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from matplotlib import cm
from PIL import Image

OUT = Path("data/spectrograms")
LABELS_CSV = Path("data/labels.csv")

DATASET_NAME = "MESD"

# columns common to all datasets (union of taxonomies)
EMOTIONS_ALL = [
    "anger", "boredom", "calm", "disgust", "fear",
    "happiness", "neutral", "sadness", "surprise",
]
MESD_EMOTIONS = {"anger", "disgust", "fear", "happiness", "neutral", "sadness"}

# 1st part of the filename (emotion)
EMOTION_CODES = {
    "Anger": "anger", "Disgust": "disgust", "Fear": "fear",
    "Happiness": "happiness", "Neutral": "neutral", "Sadness": "sadness",
}

# 2nd part of the filename (voice type) -> gender (C = child, gender unspecified)
GENDER_CODES = {"F": "female", "M": "male", "C": None}

In [ ]:
# 1. download (kagglehub manages its own cache, nothing to redo if already downloaded)
RAW = Path(kagglehub.dataset_download("saurabhshahane/mexican-emotional-speech-database-mesd"))
print("RAW =", RAW)

wavs = sorted(RAW.rglob("*.wav"))
print(len(wavs), "wav files")
print("example names:", [w.name for w in wavs[:5]])

In [ ]:
# 2. metadata (parsed from the filename) -> 1 CSV (accumulated across multiple datasets)
# standard format : <emotion>_<voice type F/M/C>_<corpus A/B>_<word>.wav
# degraded format : <emotion>_<voice type>_<corpus>_<level L1/L2>_<word>.wav (F + corpus B only)
rows = []
for wav in wavs:
    parts = wav.stem.split("_")
    if len(parts) == 4:
        emo_code, voice_type, corpus, word = parts
    elif len(parts) == 5:
        emo_code, voice_type, corpus, level, word = parts
    else:
        print("unexpected filename, skipped:", wav.name)
        continue

    rows.append({
        "file": wav.name,
        "subdir": str(wav.relative_to(RAW).parent),
        "speaker": voice_type,
        "age": None,
        "gender": GENDER_CODES.get(voice_type),
        "language": "spa",
        "emotion.confidence": None,
        "emotion.naturalness": None,
        "transcription": word.lower(),
        "duration": round(sf.info(wav).duration, 3),
        "emotion": EMOTION_CODES[emo_code.capitalize()],
    })
df = pd.DataFrame(rows)

if LABELS_CSV.exists():
    existing = pd.read_csv(LABELS_CSV, na_values=["None"])
    dup = existing.merge(
        df.assign(root=str(RAW))[["root", "file"]], on=["root", "file"], how="inner"
    )
    if len(dup):
        raise ValueError(f"{len(dup)} files from {DATASET_NAME} already present in {LABELS_CSV}")
    same = existing[existing["dataset"] == DATASET_NAME]
    prior_speaker = dict(zip(same["initial_speaker"], same["speaker"]))
    prior_transcript = dict(zip(same["initial_transcript"], same["transcript"]))
    next_speaker = int(existing["speaker"].max()) + 1
    next_transcript = int(existing["transcript"].max()) + 1
else:
    existing = None
    prior_speaker = {}
    prior_transcript = {}
    next_speaker = 1
    next_transcript = 1

df["initial_speaker"] = df["speaker"]
speaker_map = {}
for code in sorted(df["speaker"].unique()):
    if code in prior_speaker:
        speaker_map[code] = prior_speaker[code]
    else:
        speaker_map[code] = next_speaker
        next_speaker += 1
df["speaker"] = df["speaker"].map(speaker_map)

df["initial_transcript"] = df["transcription"]
transcript_map = {}
for code in sorted(df["transcription"].unique()):
    if code in prior_transcript:
        transcript_map[code] = prior_transcript[code]
    else:
        transcript_map[code] = next_transcript
        next_transcript += 1
df["transcript"] = df["transcription"].map(transcript_map)
df = df.drop(columns=["transcription"])

for emo in EMOTIONS_ALL:
    df[emo] = (df["emotion"] == emo).astype(int) if emo in MESD_EMOTIONS else None
df = df.drop(columns=["emotion"])

df.insert(0, "dataset", DATASET_NAME)
df.insert(0, "root", str(RAW))

cols = ["root", "dataset", "subdir", "file", "duration", "speaker", "initial_speaker",
        "age", "gender", "language", "emotion.confidence", "emotion.naturalness",
        "transcript", "initial_transcript"] + EMOTIONS_ALL
df = df[cols]

if existing is not None:
    df = pd.concat([existing, df], ignore_index=True)

df.to_csv(LABELS_CSV, index=False, na_rep="None")
print(len(df), "rows in", LABELS_CSV)

In [ ]:
# 3. wav -> 3s segments -> mel-spectrogram -> RGB image (accumulated across multiple datasets)
SR = 16000
DURATION = 3.0
MIN_SEG = 1.0
N_FFT = int(0.025 * SR)
HOP = int(0.010 * SR)
N_MELS = 64
N_FRAMES = int(DURATION * SR / HOP)
CHUNK = int(DURATION * SR)
MIN_SAMPLES = int(MIN_SEG * SR)

SPEC_CSV = OUT / "labels.csv"
label_by_file = dict(zip(df["file"], df[EMOTIONS_ALL].fillna(0).idxmax(axis=1)))
meta = df.set_index("file").to_dict(orient="index")

spec_existing = pd.read_csv(SPEC_CSV, na_values=["None"]) if SPEC_CSV.exists() else None
done = set(zip(spec_existing["root"], spec_existing["base_file"])) if spec_existing is not None else set()

def mel_to_rgb(y):
    y = librosa.util.fix_length(y, size=CHUNK)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel)[:, :N_FRAMES]
    z = (db - db.mean()) / (db.std() + 1e-9)
    norm = np.clip(z, -3, 3) / 6 + 0.5
    rgb = (cm.viridis(norm[::-1])[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgb)

rows = []
for wav in wavs:
    if (str(RAW), wav.name) in done:
        continue
    y, _ = librosa.load(wav, sr=SR, mono=True)
    label = label_by_file[wav.name]

    for n, start in enumerate(range(0, len(y), CHUNK)):
        seg = y[start:start + CHUNK]
        if len(seg) < MIN_SAMPLES:
            continue
        name = f"{wav.stem}_{n}"
        dest = OUT / label
        dest.mkdir(parents=True, exist_ok=True)
        mel_to_rgb(seg).save(dest / f"{name}.png")
        rows.append({
            "spectrogram": f"{label}/{name}.png",
            "base_file": wav.name,
            "chunk": n,
            **meta[wav.name],
            "duration": round(sf.info(wav).duration, 3),
        })

new_spec = pd.DataFrame(rows)
spec_df = pd.concat([spec_existing, new_spec], ignore_index=True) if spec_existing is not None else new_spec
spec_df.to_csv(SPEC_CSV, index=False, na_rep="None")
print(len(rows), "new spectrograms,", len(spec_df), "total in", SPEC_CSV)

In [ ]:
# 4. quick check
for d in sorted(OUT.iterdir()):
    if d.is_dir():
        print(d.name, len(list(d.glob("*.png"))))

display(Image.open(next(OUT.rglob("*.png"))))

import csv as _csv
with LABELS_CSV.open(encoding="utf-8") as f:
    for i, row in zip(range(5), _csv.reader(f)):
        print(row)